# 01 · Data Collection & Assessment
**Agentris — Soto Batok & Penyetan Pak Tris | A26-CS193**

Tujuan: menginventarisasi data primer, sintetis dan referensi; memeriksa kualitas sebelum cleaning. Data mentah tidak diubah. Data sintetis adalah simulasi Agustus–Oktober 2026, bukan riwayat penjualan aktual.

Sumber: [Project Plan](https://docs.google.com/document/d/1UcQiyXj9haOqE3SA-vJlvonQcbb9T-EVeyxwEkJByoE/edit), [Interview dan Use Case](https://docs.google.com/document/d/1GBdc-kcfjPI32mygfm4fEtqxjs0U5G0uu0xoefymtIU/edit), [folder sumber UMKM](https://drive.google.com/drive/folders/1HQbyc_Z9VqJN2H6lCMOOv_yhB4-eFyOe), [Pertanyaan Bisnis](https://docs.google.com/spreadsheets/d/1r84YzGIF4A3hbOdy_7r8MqazMyDEKU-fbrjmSFbH-Ps/edit).

**Unit analisis:** penjualan sintetis satu baris per item pesanan, bukan per transaksi. Pengeluaran satu baris per entri. Referensi menu adalah data primer bersih.

In [5]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Bisa dijalankan dari root proyek maupun folder notebooks.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'datasets' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari folder proyek Agentris yang sudah diekstrak.')
RAW = ROOT / 'datasets' / 'raw'
REF = ROOT / 'datasets' / 'reference'
INTERIM = ROOT / 'datasets' / 'interim'
PROCESSED = ROOT / 'datasets' / 'processed'
REPORTS = ROOT / 'reports'
for folder in [INTERIM, PROCESSED, REPORTS / 'figures']:
    folder.mkdir(parents=True, exist_ok=True)
FILES = {
    'penjualan_sintetis': RAW / 'transaksi_penjualan_sintetis_raw_gabungan.csv',
    'pengeluaran_sintetis': RAW / 'transaksi_pengeluaran_sintetis_raw.csv',
    'penjualan_primer': RAW / 'transaksi_penjualan_primer_raw.csv',
    'pengeluaran_primer': RAW / 'transaksi_pengeluaran_primer_raw.csv',
    'menu': REF / 'referensi_menu.csv',
}
def load_raw():
    return {key: pd.read_csv(path, dtype='string') for key, path in FILES.items()}
print('Project root:', ROOT)

Project root: c:\Users\hp\Downloads\Agentris_Data_Science\Agentris_Data_Science


In [6]:
data = load_raw()
for nama, df in data.items():
    print(f'{nama}: {df.shape[0]} baris, {df.shape[1]} kolom')
    display(df.head())

penjualan_sintetis: 2694 baris, 9 kolom


,id_transaksi,tanggal,nama_pembeli,jenis_makanan,jumlah,harga_satuan,subtotal,id_detail,total_transaksi
0,PJ-00001,2026-08-01,ShopeeFood,Soto ayam pisah,1,13000,13000,PJ-00001-01,13000
1,PJ-00002,2026-08-01,Pelanggan 179,Pecel lele,1,13000,13000,PJ-00002-01,16000
2,PJ-00002,2026-08-01,Pelanggan 179,Teh es,1,3000,3000,PJ-00002-02,16000
3,PJ-00003,2026-08-01,Pelanggan 146,Soto ayam,1,10000,10000,PJ-00003-01,10000
4,PJ-00004,2026-08-01,ShopeeFood,Soto ayam bungkus + nasi,1,12000,12000,PJ-00004-01,15000


pengeluaran_sintetis: 471 baris, 4 kolom


,no,tanggal,keterangan,total
0,PG-00001,2026-08-01,bumbu racik,11500
1,PG-00002,2026-08-01,Beras,17000
2,PG-00003,2026-08-01,Tempe,11000
3,PG-00004,2026-08-01,Nila,40500
4,PG-00005,2026-08-02,Minyak goreng,19000


penjualan_primer: 11 baris, 5 kolom


,no,tanggal,nama_pembeli,jenis_makanan,total
0,AS-001,Senin 07 sept,Shoopee,<NA>,10.000
1,AS-002,Senin 07 sept,Pembeli P01,<NA>,11.000
2,AS-003,Senin 07 sept,Pembeli P02,<NA>,11.000
3,AS-004,Senin 07 sept,Stpn,<NA>,40.000
4,AS-005,Senin 07 sept,Pembeli P03,<NA>,80.000


pengeluaran_primer: 13 baris, 4 kolom


,no,tanggal,keterangan,total
0,AG-001,<NA>,Lele,28
1,AG-002,<NA>,Nila,20
2,AG-003,<NA>,Ayam,46
3,AG-004,<NA>,Sapi,35
4,AG-005,<NA>,Cadar,32


menu: 23 baris, 4 kolom


,id_menu,kategori,nama_menu,harga
0,M001,Soto,Soto ayam,10000
1,M002,Soto,Soto ayam pisah,13000
2,M003,Soto,Soto sapi,14000
3,M004,Soto,Soto sapi pisah,16000
4,M005,Soto,Soto ayam bungkus + nasi,12000


## 1. Struktur, missing values dan duplikasi
Pembacaan sebagai string disengaja agar format mentah tidak hilang. Duplikasi ID transaksi penjualan bukan otomatis duplikasi baris.

In [7]:
ringkasan = []
for nama, df in data.items():
    print(nama)
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'kosong': df.isna().sum(), 'unik': df.nunique()}))
    ringkasan.append({'dataset': nama, 'baris': len(df), 'kolom': len(df.columns), 'sel_kosong': int(df.isna().sum().sum()), 'duplikat_identik': int(df.duplicated().sum())})
summary = pd.DataFrame(ringkasan)
display(summary)

penjualan_sintetis


,dtype,kosong,unik
id_transaksi,string,0,1183
tanggal,string,8,132
nama_pembeli,string,6,190
jenis_makanan,string,10,51
jumlah,string,15,5
harga_satuan,string,10,21
subtotal,string,5,31
id_detail,string,5,2646
total_transaksi,string,3,135


pengeluaran_sintetis


,dtype,kosong,unik
no,string,0,464
tanggal,string,1,109
keterangan,string,4,24
total,string,1,140


penjualan_primer


,dtype,kosong,unik
no,string,0,11
tanggal,string,0,1
nama_pembeli,string,0,10
jenis_makanan,string,11,0
total,string,0,8


pengeluaran_primer


,dtype,kosong,unik
no,string,0,13
tanggal,string,13,0
keterangan,string,0,10
total,string,0,11


menu


,dtype,kosong,unik
id_menu,string,0,23
kategori,string,0,5
nama_menu,string,0,23
harga,string,0,12


,dataset,baris,kolom,sel_kosong,duplikat_identik
0,penjualan_sintetis,2694,9,62,43
1,pengeluaran_sintetis,471,4,6,7
2,penjualan_primer,11,5,11,0
3,pengeluaran_primer,13,4,13,0
4,menu,23,4,0,0


In [8]:
jual = data['penjualan_sintetis']
belanja = data['pengeluaran_sintetis']
print('ID transaksi unik:', jual['id_transaksi'].nunique())
print('ID detail nonkosong yang berulang:')
display(jual.loc[jual.id_detail.notna() & jual.id_detail.duplicated(keep=False)].head(25))
print('Duplikat identik pengeluaran:')
display(belanja[belanja.duplicated(keep=False)].head(20))

ID transaksi unik: 1183
ID detail nonkosong yang berulang:


,id_transaksi,tanggal,nama_pembeli,jenis_makanan,jumlah,harga_satuan,subtotal,id_detail,total_transaksi
276,PJ-00119,2026-08-11,Pelanggan 012,Soto sapi,1,14000,14000,PJ-00119-01,30000
277,PJ-00119,2026-08-11,Pelanggan 012,Soto ayam pisah,1,13000,13000,PJ-00119-02,30000
278,PJ-00119,2026-08-11,Pelanggan 012,Teh es,1,3000,3000,PJ-00119-03,30000
279,PJ-00119,2026-08-11,Pelanggan 012,Soto sapi,1,14000,14000,PJ-00119-01,30000
280,PJ-00119,2026-08-11,Pelanggan 012,Soto ayam pisah,1,13000,13000,PJ-00119-02,30000
281,PJ-00119,2026-08-11,Pelanggan 012,Teh es,1,3000,3000,PJ-00119-03,30000
508,PJ-00221,2026-08-18,Pelanggan 005,Soto ayam pisah,1,13000,13000,PJ-00221-01,55000
509,PJ-00221,2026-08-18,Pelanggan 005,Soto sapi pisah,1,16000,16000,PJ-00221-02,55000
510,PJ-00221,2026-08-18,Pelanggan 005,Soto sapi,1,14000,14000,PJ-00221-03,55000
511,PJ-00221,2026-08-18,Pelanggan 005,Teh es,1,3000,3000,PJ-00221-04,55000


Duplikat identik pengeluaran:


,no,tanggal,keterangan,total
13,PG-00014,2026-08-03,Parkir,2500
14,PG-00014,2026-08-03,Parkir,2500
50,PG-00050,2026-08-10,Ayam,56500
51,PG-00050,2026-08-10,Ayam,56500
179,PG-00178,2026-09-07,Sapi,64500
180,PG-00178,2026-09-07,Sapi,64500
204,PG-00202,2026-09-12,Jeruk,15500
205,PG-00202,2026-09-12,Jeruk,15500
218,PG-00215,2026-09-14,Nila,21500
219,PG-00215,2026-09-14,Nila,21500


## 2. Format tanggal, nominal, menu dan kuantitas
Pemeriksaan berikut hanya mendeteksi anomali, belum memperbaiki nilai.

In [9]:
for nama, df in [('penjualan', jual), ('pengeluaran', belanja)]:
    print(nama, '- tanggal non-ISO atau kosong')
    display(df.loc[~df.tanggal.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False), 'tanggal'].value_counts(dropna=False))
    for col in ['jumlah', 'harga_satuan', 'subtotal', 'total_transaksi', 'total']:
        if col in df:
            print(col, '- nilai bukan bilangan bulat positif')
            display(df.loc[~df[col].str.fullmatch(r'[1-9]\d*', na=False), col].value_counts(dropna=False).head(20))
print('Nama menu tidak cocok persis dengan referensi:')
display(jual.loc[~jual.jenis_makanan.isin(data['menu'].nama_menu), 'jenis_makanan'].value_counts(dropna=False))

penjualan - tanggal non-ISO atau kosong


tanggal
32-08-2026     13
<NA>            8
04/09/2026      6
2026/08/13      5
08/08/2026      4
18/08/2026      4
07-08-2026      3
2026/08/14      3
24/08/2026      3
31 Agu 2026     3
18-09-2026      3
29/10/2026      3
10-08-2026      2
12-08-2026      2
19 Agu 2026     2
22-08-2026      2
06/09/2026      2
12/09/2026      2
2026/09/21      2
2026/10/12      2
2026/10/15      2
19/10/2026      2
2026/10/30      2
2026/08/03      1
03/08/2026      1
11 Agu 2026     1
15 Agu 2026     1
17-08-2026      1
23/08/2026      1
28/08/2026      1
2 Sep 2026      1
10-09-2026      1
19/09/2026      1
2026/09/22      1
26-09-2026      1
2026/09/28      1
30-09-2026      1
30/09/2026      1
11/10/2026      1
24 Okt 2026     1
Name: count, dtype: int64[pyarrow]

jumlah - nilai bukan bilangan bulat positif


jumlah
<NA>    15
0        5
Name: count, dtype: int64[pyarrow]

harga_satuan - nilai bukan bilangan bulat positif


harga_satuan
<NA>         10
Rp 3.000      5
Rp 17.000     4
Rp 20.000     3
Rp 10.000     2
Rp 12.000     2
Rp 6.000      1
Rp 7.000      1
Rp 13.000     1
Rp 14.000     1
Name: count, dtype: int64[pyarrow]

... (6 outputs left out)

total - nilai bukan bilangan bulat positif


total
-35000       1
1 rb         1
-43500       1
Rp 28.500    1
37.5 rb      1
Rp 61.000    1
 30500       1
0            1
Rp 15.000    1
Rp 37.000    1
57.5 rb      1
Rp 22.000    1
41.5 rb      1
Rp 30.000    1
<NA>         1
 24000       1
 14000       1
 36500       1
21.500       1
2.500        1
Name: count, dtype: int64[pyarrow]

Nama menu tidak cocok persis dengan referensi:


jenis_makanan
<NA>                              10
jeruk es                           6
teh es                             5
sotto ayam                         4
sotto ayam pisah                   3
sotto daging bungkus + nasi        3
ayam sambal                        3
NASI GORENG TELUR                  3
sotto sapi pisah                   3
  Choco crunchy                    2
sotto ayam bungkus + nasi          2
AYAM SAMBAL                        2
  Jeruk es                         2
soto daging bungkus + nasi         2
  Soto ayam                        2
  Soto daging bungkus + nasi       1
  Pecel lele                       1
TEH ES                             1
sotto sapi                         1
JERUK PANAS                        1
  Nasi goreng teri hijau           1
soto sapi pisah                    1
donat polos / gula halus           1
pecel lele                         1
  Nasi goreng telur                1
NILA SAMBAL                        1
JERUK ES                

In [10]:
print('Variasi keterangan pengeluaran:')
display(belanja.keterangan.value_counts(dropna=False))
print('Transaksi tanpa ID detail:')
display(jual[jual.id_detail.isna()])

Variasi keterangan pengeluaran:


keterangan
Beras             40
Telur             40
Bumbu racik       40
Tempe             36
Minyak goreng     35
Bahan donat       33
Parkir            31
Jeruk             31
Sapi              30
Sayuran           30
Nila              28
Ayam              28
Lele              27
Kemasan           24
<NA>               4
 sayuran           3
 sapi              2
 parkir            2
 bumbu racik       1
 nila              1
 ayam              1
 jeruk             1
 bahan donat       1
 kemasan           1
 beras             1
Name: count, dtype: int64[pyarrow]

Transaksi tanpa ID detail:


,id_transaksi,tanggal,nama_pembeli,jenis_makanan,jumlah,harga_satuan,subtotal,id_detail,total_transaksi
492,PJ-00214,2026-08-18,Pelanggan 054,<NA>,<NA>,<NA>,<NA>,<NA>,3000
914,PJ-00403,2026-08-31,Pelanggan 002,<NA>,<NA>,<NA>,<NA>,<NA>,26000
1029,PJ-00451,2026-09-04,Pelanggan 101,<NA>,<NA>,<NA>,<NA>,<NA>,10000
1412,PJ-00618,2026-09-17,Pelanggan 083,<NA>,<NA>,<NA>,<NA>,<NA>,29000
1575,PJ-00689,2026-09-23,Pelanggan 073,<NA>,<NA>,<NA>,<NA>,<NA>,23000


## 3. Keterbatasan data primer
Penjualan WhatsApp: 11 entri, total acuan Rp285.000; tahun dan rincian item tidak tercantum. Jumlah entri belum tentu sama dengan jumlah transaksi. Pengeluaran: 13 entri, total acuan Rp269.000; tanggal belum tersedia, satuan ribuan rupiah perlu dikonfirmasi. Entri bahan yang berulang tidak otomatis duplikat.

**TODO:** tulis temuan assessment per dataset, keputusan penanganan yang akan diambil dan hal yang perlu dikonfirmasi ke pemilik. Jangan memasukkan data primer ke rekap bulanan tanpa tanggal lengkap.

In [12]:
display(data['penjualan_primer'])
display(data['pengeluaran_primer'])
summary.to_csv(REPORTS / 'assessment_summary.csv', index=False)

,no,tanggal,nama_pembeli,jenis_makanan,total
0,AS-001,Senin 07 sept,Shoopee,<NA>,10.000
1,AS-002,Senin 07 sept,Pembeli P01,<NA>,11.000
2,AS-003,Senin 07 sept,Pembeli P02,<NA>,11.000
3,AS-004,Senin 07 sept,Stpn,<NA>,40.000
4,AS-005,Senin 07 sept,Pembeli P03,<NA>,80.000
5,AS-006,Senin 07 sept,Pembeli P04,<NA>,25.000
6,AS-007,Senin 07 sept,Ojek,<NA>,17.000
7,AS-008,Senin 07 sept,Ojek,<NA>,17.000
8,AS-009,Senin 07 sept,Pembeli P05,<NA>,17.000
9,AS-010,Senin 07 sept,Pembeli P06,<NA>,42.000


,no,tanggal,keterangan,total
0,AG-001,<NA>,Lele,28
1,AG-002,<NA>,Nila,20
2,AG-003,<NA>,Ayam,46
3,AG-004,<NA>,Sapi,35
4,AG-005,<NA>,Cadar,32
5,AG-006,<NA>,Jeeuk,10
6,AG-007,<NA>,Puyuh,13.5
7,AG-008,<NA>,Racik,4
8,AG-009,<NA>,Parkir,1
9,AG-010,<NA>,Tempe,7
